# 7.5 · Backtracking

*Algorithm design* · 7 exercises

Build answers one choice at a time, undo the choice, try the next: orderings, combinations, placements and puzzles, with pruning that cuts dead branches early.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-7.5) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-7.5) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-7.5)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 7.5.1 · Every ordering  `Core`

**Task.** Write `orderings(items)` that returns every ordering of the distinct `items`. Build one path: at each step try every unused item in input order, recurse, then undo the choice.

**Example.** `[1, 2, 3]` -> `[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]`

**Big O.** O(N * N!) time: N! orderings of N items each, plus O(N) for the path and the used flags.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-5-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-5-1)

In [ ]:
# 7.5.1 Every ordering
def orderings(items):
    raise NotImplementedError

In [ ]:
def test_ex_7_5_1():
    assert orderings([1, 2, 3]) == [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]
    assert orderings([]) == [[]]
    assert orderings([7]) == [[7]]
    assert len(orderings([1, 2, 3, 4, 5])) == 120


test_ex_7_5_1()
print('7.5.1 passed')

### 7.5.2 · Letters for a keypad number  `Core`

**Task.** On a phone keypad `2` is `abc`, `3` `def`, `4` `ghi`, `5` `jkl`, `6` `mno`, `7` `pqrs`, `8` `tuv` and `9` `wxyz`. Write `keypad_letters(digits)` that returns every letter string the digits could type, in keypad order (`[]` for no digits). Choose a letter for each digit in turn, recursing on the rest.

**Example.** `"23"` -> `["ad", "ae", "af", "bd", "be", "bf", "cd", "ce", "cf"]`

**Big O.** O(4^N * N) for N digits: up to four letters per digit, and each finished string costs O(N) to build.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-5-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-5-2)

In [ ]:
# 7.5.2 Letters for a keypad number
def keypad_letters(digits):
    raise NotImplementedError

In [ ]:
def test_ex_7_5_2():
    assert keypad_letters('23') == ['ad', 'ae', 'af', 'bd', 'be', 'bf', 'cd', 'ce', 'cf']
    assert keypad_letters('') == []
    assert keypad_letters('7') == ['p', 'q', 'r', 's']
    assert len(keypad_letters('79')) == 16
    assert keypad_letters('99')[-1] == 'zz'


test_ex_7_5_2()
print('7.5.2 passed')

### 7.5.3 · Combinations that hit a target  `Practice`

**Task.** `candidates` are distinct positive integers, each usable any number of times. Write `combinations_to(candidates, target)` (`target > 0`) that returns every combination summing to `target`, each in ascending order and the list in dictionary order. Sort, then recurse from the current candidate onward, stopping once a candidate exceeds what is left.

**Example.** `[3, 4, 5], 11` -> `[[3, 3, 5], [3, 4, 4]]`

**Big O.** Exponential in the worst case: sorting lets a branch stop as soon as the next candidate overshoots, and only taking candidates at or after the last one avoids repeats.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-5-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-5-3)

In [ ]:
# 7.5.3 Combinations that hit a target
def combinations_to(candidates, target):
    raise NotImplementedError

In [ ]:
def test_ex_7_5_3():
    assert combinations_to([3, 4, 5], 11) == [[3, 3, 5], [3, 4, 4]]
    assert combinations_to([2], 1) == []
    assert combinations_to([1], 3) == [[1, 1, 1]]
    assert combinations_to([5, 2], 7) == [[2, 5]]
    assert len(combinations_to([1, 2], 6)) == 4


test_ex_7_5_3()
print('7.5.3 passed')

### 7.5.4 · Queens that do not attack  `Practice`

**Task.** Write `queen_placements(n)` that counts the ways to put `n` queens on an `n` by `n` board so that no two share a row, column or diagonal. Place one queen per row; track the used columns, `row - col` diagonals and `row + col` diagonals in sets.

**Example.** `4` -> `2`

**Big O.** O(N!) worst case, far less in practice: one queen per row, and sets of used columns and diagonals reject a square in O(1).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-5-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-5-4)

In [ ]:
# 7.5.4 Queens that do not attack
def queen_placements(n):
    raise NotImplementedError

In [ ]:
def test_ex_7_5_4():
    assert [queen_placements(n) for n in (4, 1, 2, 3, 5, 6, 8)] == [2, 1, 0, 0, 10, 4, 92]


test_ex_7_5_4()
print('7.5.4 passed')

### 7.5.5 · Subsets when values repeat  `Practice`

**Task.** Write `unique_subsets(items)` that returns every distinct subset of `items`, which may repeat values. Sort first; record the current path at every call, and when choosing the next item at one depth skip a value equal to the one just tried. Each subset is ascending, listed in the order this walk finds them.

**Example.** `[2, 1, 2]` -> `[[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]`

**Big O.** O(N * 2^N) worst case: sorting puts equal values side by side, and skipping a repeat at the same depth stops the same subset being built twice.

**Note.** Compare 7.1.18, which assumes distinct items.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-5-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-5-5)

In [ ]:
# 7.5.5 Subsets when values repeat
def unique_subsets(items):
    raise NotImplementedError

In [ ]:
def test_ex_7_5_5():
    assert unique_subsets([2, 1, 2]) == [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]
    assert unique_subsets([]) == [[]]
    assert unique_subsets([3, 3]) == [[], [3], [3, 3]]
    assert len(unique_subsets([1, 2, 3])) == 8
    assert len(unique_subsets([1, 1, 2, 2])) == 9


test_ex_7_5_5()
print('7.5.5 passed')

### 7.5.6 · Split digits into an address  `Practice`

**Task.** Write `address_splits(digits)` that returns every way to insert three dots into `digits` so that the four parts are each `0` to `255` with no leading zero (`"0"` is fine, `"01"` is not), joined with `.`. Recurse part by part, trying lengths 1, 2 and 3, and stop early when too many or too few digits are left.

**Example.** `"101023"` -> `["1.0.10.23", "1.0.102.3", "10.1.0.23", "10.10.2.3", "101.0.2.3"]`

**Big O.** O(1) for a fixed four parts of up to three digits: at most 3^4 splits are tried, each checked in constant time.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-5-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-5-6)

In [ ]:
# 7.5.6 Split digits into an address
def address_splits(digits):
    raise NotImplementedError

In [ ]:
def test_ex_7_5_6():
    assert address_splits('101023') == ['1.0.10.23', '1.0.102.3', '10.1.0.23', '10.10.2.3', '101.0.2.3']
    assert address_splits('0000') == ['0.0.0.0']
    assert address_splits('1111') == ['1.1.1.1']
    assert address_splits('123') == []
    assert address_splits('256256256256') == []
    assert address_splits('255255255255') == ['255.255.255.255']


test_ex_7_5_6()
print('7.5.6 passed')

### 7.5.7 · Sudoku solver  `Stretch`

**Task.** `board` is 9 lists of 9 digits, `0` for empty. Write `solve_sudoku(board)` that fills the empty cells in place so every row, column and 3 by 3 box holds 1 to 9 once, returning `True`, or returns `False` (board unchanged) when no filling exists. Keep a bit mask of used digits per row, column and box; always fill the empty cell with the fewest candidates next, and undo on failure.

**Example.** a full grid with 45 cells blanked -> `True`, and every row, column and box holds 1 to 9

**Big O.** Exponential in the worst case, fast on real puzzles: bit masks per row, column and box make each candidate check O(1), and filling the most constrained cell first prunes hardest.

**Note.** 2.7.7 checks a board; this one fills it.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-7-5-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-7-5-7)

In [ ]:
# 7.5.7 Sudoku solver
def solve_sudoku(board):
    raise NotImplementedError

In [ ]:
def test_ex_7_5_7():

    def solved(b):
        groups = [b[r] for r in range(9)] + [[b[r][c] for r in range(9)] for c in range(9)]
        groups += [[b[r][c] for r in range(br, br + 3) for c in range(bc, bc + 3)] for br in (0, 3, 6) for bc in (0, 3, 6)]
        return all((sorted(g) == list(range(1, 10)) for g in groups))
    full = [[(r * 3 + r // 3 + c) % 9 + 1 for c in range(9)] for r in range(9)]
    puzzle = [[0 if (r * 7 + c * 3) % 9 < 5 else full[r][c] for c in range(9)] for r in range(9)]
    given = [row[:] for row in puzzle]
    assert sum((row.count(0) for row in puzzle)) == 45
    assert solve_sudoku(puzzle)
    assert solved(puzzle)
    assert all((given[r][c] in (0, puzzle[r][c]) for r in range(9) for c in range(9)))
    stuck = [[0] * 9 for _ in range(9)]
    stuck[0][:8] = [1, 2, 3, 4, 5, 6, 7, 8]
    stuck[1][8] = 9
    before = [row[:] for row in stuck]
    assert not solve_sudoku(stuck)
    assert stuck == before
    empty = [[0] * 9 for _ in range(9)]
    assert solve_sudoku(empty)
    assert solved(empty)


test_ex_7_5_7()
print('7.5.7 passed')